# 02 — Exploratory Data Analysis
EDA over `data/processed/resumes.csv`. All artifacts go to `reports/eda/`.
No TF-IDF fitting on the full dataset for modeling is exported — class-keyword TF-IDF below is used only for visualization of category-distinguishing terms.

In [1]:
from pathlib import Path
import sys, re
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
EDA = ROOT / 'reports' / 'eda'
EDA.mkdir(parents=True, exist_ok=True)

import pandas as pd, numpy as np, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
df = pd.read_csv(ROOT / 'data' / 'processed' / 'resumes.csv')
df['text'] = df['text'].fillna('')
df['word_count'] = df['text'].apply(lambda t: len(t.split()))
df['char_count'] = df['text'].str.len()
print(df.shape)

(2500, 6)


## A. Class distribution

In [2]:
counts = df['category'].value_counts().sort_values(ascending=False)
plt.figure(figsize=(14,6))
plt.bar(counts.index, counts.values)
plt.xticks(rotation=90)
plt.title('Resume Class Distribution')
plt.tight_layout()
plt.savefig(EDA / 'class_distribution.png', dpi=120)
plt.show()
print(counts)

category
FINANCE                   134
BUSINESS-DEVELOPMENT      120
INFORMATION-TECHNOLOGY    120
ACCOUNTANT                118
CHEF                      118
ADVOCATE                  118
ENGINEERING               118
AVIATION                  117
FITNESS                   117
SALES                     116
HEALTHCARE                115
CONSULTANT                115
BANKING                   115
CONSTRUCTION              112
PUBLIC-RELATIONS          111
HR                        110
DESIGNER                  107
ARTS                      103
TEACHER                   102
APPAREL                    97
DIGITAL-MEDIA              96
AGRICULTURE                63
AUTOMOBILE                 36
BPO                        22
Name: count, dtype: int64


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\1795224983.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## B. Resume word count distribution

In [3]:
plt.figure(figsize=(10,5))
plt.hist(df['word_count'], bins=60)
plt.axvline(df['word_count'].median(), color='red', linestyle='--', label=f"median={df['word_count'].median():.0f}")
plt.title('Resume Word Count Distribution')
plt.legend(); plt.tight_layout()
plt.savefig(EDA / 'resume_length.png', dpi=120)
plt.show()
print(df['word_count'].describe())

count    2500.000000
mean      811.811600
std       361.398777
min         0.000000
25%       652.000000
50%       759.000000
75%       934.250000
max      5190.000000
Name: word_count, dtype: float64


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\4214973254.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## C. Character count distribution

In [4]:
plt.figure(figsize=(10,5))
plt.hist(df['char_count'], bins=60, color='orange')
plt.axvline(df['char_count'].median(), color='red', linestyle='--', label=f"median={df['char_count'].median():.0f}")
plt.title('Resume Character Count Distribution')
plt.legend(); plt.tight_layout()
plt.savefig(EDA / 'character_length.png', dpi=120)
plt.show()
print(df['char_count'].describe())

count     2500.000000
mean      5929.745600
std       2621.817877
min          0.000000
25%       4826.000000
50%       5554.000000
75%       6855.250000
max      35123.000000
Name: char_count, dtype: float64


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\923291215.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## D/E/F. Top words, bigrams, trigrams
Preprocessing: lowercase, remove URLs/emails, strip markup; filter English stopwords plus common resume boilerplate (experience, skills, responsible, including, using, ...) so analysis isn't dominated by template words.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

BOILERPLATE = {'experience','work','working','skills','skill','responsible','including','using','used','use','strong','good','excellent','knowledge','ability','years','year','team','teams','ability','development','management','customer','support','service','company','position','role','opportunity','professional','environment','responsibilities','required','prefer','etc','provide','provided','also','well','across','within','various','multiple','new','based','including','other','such','like','may','one','two','three','high','low','least','ensure','ensured','assist','assisted','performed','perform','created','developed','maintained','managed','responsible'}

STOP = set(__import__('ml.preprocessing', fromlist=['BASIC_STOPWORDS']).BASIC_STOPWORDS) | BOILERPLATE
texts = df['text'].str.lower()
texts = texts.str.replace(r'https?://\S+|www\.\S+', ' ', regex=True)
texts = texts.str.replace(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b', ' ', regex=True)
texts = texts.str.replace(r'[^a-z0-9\+#\s\.]', ' ', regex=True)

def top_ngrams(n, topn=25):
    vec = CountVectorizer(ngram_range=(n,n), stop_words=list(STOP), min_df=5, max_df=0.8)
    X = vec.fit_transform(texts)
    freq = X.sum(axis=0).A1
    terms = vec.get_feature_names_out()
    order = np.argsort(freq)[::-1][:topn]
    return pd.Series(freq[order], index=terms[order])

for n, name, fname in [(1,'Top Words','top_words.png'), (2,'Top Bigrams','top_bigrams.png'), (3,'Top Trigrams','top_trigrams.png')]:
    s = top_ngrams(n)
    plt.figure(figsize=(12,6))
    plt.barh(s.index[::-1], s.values[::-1])
    plt.title(name); plt.tight_layout()
    plt.savefig(EDA / fname, dpi=120); plt.show()

C:\Users\JIYA SADARIA\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['aren', 'can', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'let', 'll', 'mustn', 're', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\3002280751.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(EDA / fname, dpi=120); plt.show()
C:\Users\JIYA SADARIA\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['aren', 'can', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'let', 'll', 'mustn', 're', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\3002280751.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(EDA / fname, dpi=120); plt.show()
C:\Users\JIYA SADARIA\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['aren', 'can', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'let', 'll', 'mustn', 're', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\3002280751.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(EDA / fname, dpi=120); plt.show()


## G. WordCloud (supplementary)

In [6]:
from wordcloud import WordCloud
corpus = ' '.join(texts.tolist())
wc = WordCloud(width=1200, height=600, background_color='white', max_words=200).generate(corpus)
plt.figure(figsize=(14,7))
plt.imshow(wc); plt.axis('off'); plt.title('Resume WordCloud'); plt.tight_layout()
plt.savefig(EDA / 'wordcloud.png', dpi=120); plt.show()

C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\2869715892.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.savefig(EDA / 'wordcloud.png', dpi=120); plt.show()


## H. Class-wise distinguishing terms (TF-IDF, visualization only)

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000, stop_words=list(STOP), ngram_range=(1,1))
X = tf.fit_transform(texts)
terms = np.array(tf.get_feature_names_out())
cats = df['category'].values
mean_tfidf = pd.Series(dtype=float)
rows = {}
for cat in sorted(set(cats)):
    rows[cat] = X[cats == cat].mean(axis=0).A1
top_terms = {}
for cat, arr in rows.items():
    top_terms[cat] = terms[np.argsort(arr)[::-1][:10]].tolist()
for cat in list(top_terms)[:8]:
    print(cat, '->', ', '.join(top_terms[cat]))

# plot top terms per a few representative classes
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, cat in zip(axes.ravel(), list(top_terms)[:8]):
    arr = rows[cat]
    idx = np.argsort(arr)[::-1][:10]
    ax.barh(terms[idx][::-1], arr[idx][::-1])
    ax.set_title(cat)
plt.tight_layout()
plt.savefig(EDA / 'class_keywords.png', dpi=120)
plt.show()

C:\Users\JIYA SADARIA\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:402: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['aren', 'can', 'couldn', 'didn', 'doesn', 'don', 'hadn', 'hasn', 'haven', 'isn', 'let', 'll', 'mustn', 're', 'shan', 'shouldn', 've', 'wasn', 'weren', 'won', 'wouldn'] not in stop_words.
  warnings.warn(


ACCOUNTANT -> accounting, financial, accountant, accounts, tax, ledger, payroll, statements, monthly, state
ADVOCATE -> advocate, state, city, patient, care, name, health, sales, medical, patients
AGRICULTURE -> agriculture, state, city, research, name, program, students, project, data, health
APPAREL -> sales, store, state, city, apparel, product, name, merchandising, merchandise, manager
ARTS -> students, arts, state, city, school, student, teacher, name, art, learning
AUTOMOBILE -> claims, state, city, name, sales, call, automobile, customers, data, insurance
AVIATION -> aviation, aircraft, maintenance, flight, state, safety, city, equipment, training, systems
BANKING -> banking, financial, business, sales, loan, bank, state, city, client, mortgage


C:\Users\JIYA SADARIA\AppData\Local\Temp\ipykernel_21740\3034407622.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Quality report regeneration

In [8]:
import subprocess
r = subprocess.run([sys.executable, str(ROOT / 'scripts' / 'data' / 'analyze_quality.py')], cwd=ROOT)
print('exit code:', r.returncode)
print((ROOT / 'reports' / 'eda' / 'data_quality.txt').read_text())

exit code: 0
Dataset Summary
---------------
Total resumes: 2500
Number of classes: 24
Largest class: FINANCE (134)
Smallest class: BPO (22)
Missing texts: 1
Empty texts: 1
Very short resumes (<50 words): 1
Duplicate filenames: 0
Exact duplicate texts: 18
Duplicate groups: 18

Text Length (words)
-------------------
Minimum: 0
Maximum: 5190
Median: 759
Mean: 811.81

Text Length (characters)
------------------------
Minimum: 0
Maximum: 35123
Median: 5554
Mean: 5929.75

Class Distribution
------------------
ACCOUNTANT: 118
ADVOCATE: 118
AGRICULTURE: 63
APPAREL: 97
ARTS: 103
AUTOMOBILE: 36
AVIATION: 117
BANKING: 115
BPO: 22
BUSINESS-DEVELOPMENT: 120
CHEF: 118
CONSTRUCTION: 112
CONSULTANT: 115
DESIGNER: 107
DIGITAL-MEDIA: 96
ENGINEERING: 118
FINANCE: 134
FITNESS: 117
HEALTHCARE: 115
HR: 110
INFORMATION-TECHNOLOGY: 120
PUBLIC-RELATIONS: 111
SALES: 116
TEACHER: 102

Class Imbalance Note
--------------------
Max/min class-size ratio: 6.09. Minority classes are reported for Member 2; no resamp